# Nasdaq Futures (NQ) Market Analysis & Trading Strategy
## Time-Series Analysis and Backtesting Project (2025)

### Prepared By:
Obaida Aladday

---

## Project Objective

This project analyzes real historical Nasdaq Futures (NQ) market data using two different timeframes:

- 1-Minute Data (1m)
- 15-Minute Data (15m)

The analysis focuses exclusively on 2025 market activity.

The project includes:

### 1. Data Processing
- Timestamp handling
- Chronological sorting
- Missing values inspection
- Duplicate validation
- Dataset integrity checks

### 2. Time-Series Analysis
- Returns analysis
- Volatility analysis
- Stationarity testing (ADF Test)
- Autocorrelation analysis
- Statistical distribution analysis
- Market behavior comparison between 1m and 15m

### 3. Trading Strategy & Backtesting
A simple trend-following strategy based on:

- EMA50 trend direction
- Volume confirmation

Performance evaluation includes:

- Cumulative Return
- Maximum Drawdown
- Win Rate
- Total Trades

---

## Goal

The objective of this project is not predicting the market perfectly, but rather demonstrating:

- Financial reasoning
- Time-series understanding
- Analytical thinking
- Clean data workflow
- Strategy evaluation methodology

In [ ]:
# ==================================================
# Libraries
# ==================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

import scipy.stats as stats

sns.set(style="whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)

pd.set_option("display.max_columns", None)

# Step 1 — Data Loading

The datasets contain historical Nasdaq Futures (NQ) trading data across two timeframes:

### 1-Minute Dataset
Contains highly granular market activity.

### 15-Minute Dataset
Provides a smoother view of market trends.

The first step is loading both datasets and inspecting their dimensions.

In [ ]:
# ==================================================
# Load Data
# ==================================================

from google.colab import drive

drive.mount('/content/drive')

df_1m = pd.read_csv(
    "/content/drive/MyDrive/NQ_Data/1min.csv",
    low_memory=False
)

df_15m = pd.read_csv(
    "/content/drive/MyDrive/NQ_Data/NQ_15min.csv",
    low_memory=False
)

print("1-Minute Dataset Shape:", df_1m.shape)
print("15-Minute Dataset Shape:", df_15m.shape)

# Step 1 — Data Processing

Financial time-series analysis requires properly formatted timestamps and chronological ordering.

The following preprocessing steps are performed:

### Timestamp Handling
Convert timestamps into datetime format.

### Chronological Sorting
Ensure market observations follow correct time order.

### Data Filtering
Only observations from 2025 are selected.

This ensures consistency for financial analysis and strategy development.

In [ ]:
# ==================================================
# Timestamp Handling
# ==================================================

df_15m['timestamps'] = pd.to_datetime(
    df_15m['timestamps']
)

df_1m['timestamp'] = pd.to_datetime(
    df_1m['Date'] + ' ' + df_1m['Time']
)

df_1m.drop(
    columns=['Date', 'Time'],
    inplace=True
)

# ==================================================
# Chronological Sorting
# ==================================================

df_1m = df_1m.sort_values(
    'timestamp'
).reset_index(drop=True)

df_15m = df_15m.sort_values(
    'timestamps'
).reset_index(drop=True)

# ==================================================
# Filter 2025 Only
# ==================================================

df_1m_2025 = df_1m[
    df_1m['timestamp'].dt.year == 2025
].copy()

df_15m_2025 = df_15m[
    df_15m['timestamps'].dt.year == 2025
].copy()

In [ ]:
# ==================================================
# Data Validation & Integrity Checks
# ==================================================

validation_summary = pd.DataFrame({

    "Dataset": ["1m", "15m"],

    "Missing Values": [
        df_1m_2025.isnull().sum().sum(),
        df_15m_2025.isnull().sum().sum()
    ],

    "Duplicate Rows": [
        df_1m_2025.duplicated().sum(),
        df_15m_2025.duplicated().sum()
    ],

    "Chronological Order": [
        df_1m_2025['timestamp'].is_monotonic_increasing,
        df_15m_2025['timestamps'].is_monotonic_increasing
    ]
})

display(validation_summary)

### Data Processing Summary

The datasets were successfully prepared for analysis.

Key observations:

- Only 2025 observations were selected.
- Timestamp conversion was completed successfully.
- Data was sorted chronologically.
- No missing values were detected.
- No duplicate observations were found.
- Time-series consistency was verified.

This indicates that both datasets are clean and suitable for financial analysis.

In [ ]:
# ==================================================
# Dataset Overview
# ==================================================

overview_1m = pd.DataFrame({

    "Column Type":
    df_1m_2025.dtypes.astype(str),

    "Missing Values":
    df_1m_2025.isnull().sum(),
})

overview_15m = pd.DataFrame({

    "Column Type":
    df_15m_2025.dtypes.astype(str),

    "Missing Values":
    df_15m_2025.isnull().sum(),
})

print("1m Dataset Overview")
display(overview_1m)

print("15m Dataset Overview")
display(overview_15m)

# Statistical Summary

Basic descriptive statistics are calculated to understand:

- Price distribution
- Trading activity
- Market variability
- Price range dynamics

This provides a high-level understanding of market behavior before deeper time-series analysis.

In [ ]:
# ==================================================
# Statistical Summary
# ==================================================

display(
    df_1m_2025[
        ['Open', 'High', 'Low', 'Close', 'Volume']
    ].describe()
)

display(
    df_15m_2025[
        ['open', 'high', 'low', 'close', 'volume']
    ].describe()
)

### Key Observations

- Price levels vary significantly across the year, reflecting different market conditions and trend regimes.

- Trading volume exhibits high variability, suggesting alternating periods of low and intense market activity.

- Similar statistical characteristics between the 1-minute and 15-minute datasets indicate strong consistency across timeframes.

These findings confirm the reliability of the datasets for time-series modeling and strategy development.

# Step 2 — Time-Series Analysis

Financial markets are inherently dynamic and time-dependent.

This section focuses on analyzing market behavior through:

### Feature Engineering
Creating financial indicators such as:

- Returns
- Volatility
- Momentum
- Moving averages
- Volume behavior

### Statistical Testing
Using the Augmented Dickey-Fuller (ADF) Test to evaluate stationarity.

### Market Behavior Analysis
Studying:

- Volatility clustering
- Correlation structure
- Return distributions
- Tail risks
- Autocorrelation patterns

The objective is to better understand market structure before developing a trading strategy.

In [ ]:
# ==================================================
# Feature Engineering
# ==================================================

# ---------- 1m Dataset ----------

df_1m_2025['returns'] = np.log(
    df_1m_2025['Close'] /
    df_1m_2025['Close'].shift(1)
)

df_1m_2025['volatility_30'] = (
    df_1m_2025['returns']
    .rolling(30)
    .std()
)

df_1m_2025['momentum_10'] = (
    df_1m_2025['Close']
    .pct_change(10)
)

df_1m_2025['ma_fast'] = (
    df_1m_2025['Close']
    .rolling(10)
    .mean()
)

df_1m_2025['ma_slow'] = (
    df_1m_2025['Close']
    .rolling(50)
    .mean()
)

df_1m_2025['volume_ma'] = (
    df_1m_2025['Inc Vol']
    .rolling(30)
    .mean()
)

df_1m_2025['volume_spike'] = (
    df_1m_2025['Inc Vol']
    >
    df_1m_2025['volume_ma'] * 1.5
)

# ---------- 15m Dataset ----------

df_15m_2025['returns'] = np.log(
    df_15m_2025['close'] /
    df_15m_2025['close'].shift(1)
)

df_15m_2025['volatility_30'] = (
    df_15m_2025['returns']
    .rolling(30)
    .std()
)

df_15m_2025['momentum_10'] = (
    df_15m_2025['close']
    .pct_change(10)
)

df_15m_2025['ma_fast'] = (
    df_15m_2025['close']
    .rolling(10)
    .mean()
)

df_15m_2025['ma_slow'] = (
    df_15m_2025['close']
    .rolling(50)
    .mean()
)

print("Feature Engineering Completed Successfully.")

### Why Were These Features Created?

Several financial indicators were engineered to better understand market behavior.

#### Returns
Measure short-term price movement and market performance.

#### Volatility
Measures market uncertainty and risk.

#### Momentum
Captures short-term directional strength.

#### Moving Averages
Used to identify market trends.

#### Volume Features
Help identify unusual trading activity and liquidity spikes.

These indicators are commonly used in quantitative finance and trading systems.

In [ ]:
# ==================================================
# ADF Test
# ==================================================

adf_results = []

def collect_adf(series, name):

    result = adfuller(
        series.dropna()
    )

    adf_results.append({

        "Series": name,

        "ADF Statistic":
        result[0],

        "P-Value":
        result[1],

        "Stationary":
        result[1] < 0.05
    })

collect_adf(
    df_1m_2025['Close'],
    '1m Close Price'
)

collect_adf(
    df_15m_2025['close'],
    '15m Close Price'
)

collect_adf(
    df_1m_2025['returns'],
    '1m Returns'
)

collect_adf(
    df_15m_2025['returns'],
    '15m Returns'
)

adf_summary = pd.DataFrame(
    adf_results
)

display(adf_summary)

### ADF Test Interpretation

The ADF test evaluates whether a time series is statistically stationary.

### Findings

- Close prices were found to be non-stationary, which is expected in financial markets because prices tend to follow trends over time.

- Returns were found to be stationary, making them more suitable for statistical modeling and trading analysis.
This behavior is consistent with financial theory and supports using returns instead of raw prices for modeling.

In [ ]:
fig = px.line(
    df_1m_2025,
    x='timestamp',
    y='returns',
    title='1-Minute Returns Over Time',
    template='plotly_white'
)

fig.show()

### Insight

Returns remain centered around zero for most periods, with occasional sharp spikes.

This behavior suggests that the market alternates between:

- Long calm periods
- Short bursts of intense activity

Such movements indicate volatility clustering and highlight the importance of dynamic risk management.

In [ ]:
fig = px.line(
    df_1m_2025,
    x='timestamp',
    y='volatility_30',
    title='Rolling Volatility (30) – 1m',
    template='plotly_white'
)

fig.show()

### Insight

The market demonstrates clear volatility clustering behavior.

Periods of stability are followed by sudden increases in volatility, reflecting changes in liquidity, trader activity, or major market events.

This suggests that market risk is dynamic rather than constant.

In [ ]:
corr_matrix = df_1m_2025[
    [
        'returns',
        'volatility_30',
        'Volume',
        'Inc Vol'
    ]
].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu',
    title='Feature Correlation Heatmap'
)

fig.show()

### Insight

The correlation matrix reveals weak linear relationships between returns and trading volume.

This indicates that increased trading activity alone does not necessarily guarantee larger price movements.

Market behavior appears influenced by multiple interacting factors rather than a single variable.

In [ ]:
fig = px.histogram(
    df_1m_2025,
    x='returns',
    nbins=120,
    marginal='box',
    title='Returns Distribution',
    template='plotly_white'
)

fig.show()

### Insight

The distribution of returns deviates significantly from a normal distribution.

Extreme observations occur more frequently than expected, suggesting elevated tail risk and the possibility of sudden market movements.

In [ ]:
plt.figure(figsize=(10,5))

plot_acf(
    df_1m_2025['returns'].dropna(),
    lags=50
)

plt.title("Autocorrelation of Returns")

plt.show()

### Insight

The autocorrelation structure suggests limited predictability in short-term returns.

Most lag relationships remain weak, supporting the idea that financial markets exhibit near-random short-term behavior.

In [ ]:
# ==================================================
# 15m Returns Over Time
# ==================================================

fig = px.line(
    df_15m_2025,
    x='timestamps',
    y='returns',
    title='15-Minute Returns Over Time',
    template='plotly_white'
)

fig.show()

In [ ]:
# ==================================================
# 15m Rolling Volatility
# ==================================================

fig = px.line(
    df_15m_2025,
    x='timestamps',
    y='volatility_30',
    title='15-Minute Rolling Volatility',
    template='plotly_white'
)

fig.show()

In [ ]:
kurt = stats.kurtosis(
    df_1m_2025['returns'].dropna()
)

skew = stats.skew(
    df_1m_2025['returns'].dropna()
)

print("Kurtosis:", kurt)
print("Skewness:", skew)

### Statistical Distribution Characteristics

#### Kurtosis

A very high kurtosis value indicates fat tails, meaning extreme market movements occur more frequently than expected under a normal distribution.

#### Skewness

Negative skewness indicates downside movements tend to be stronger than upside movements.

This suggests that sudden downward market shocks are more severe than upward spikes.

In [ ]:
comparison_stats = pd.DataFrame({

    "Metric": [
        "Mean Return",
        "Std Return",
        "Mean Volatility",
        "Max Volatility"
    ],

    "1m": [

        df_1m_2025['returns'].mean(),

        df_1m_2025['returns'].std(),

        df_1m_2025['volatility_30'].mean(),

        df_1m_2025['volatility_30'].max()
    ],

    "15m": [

        df_15m_2025['returns'].mean(),

        df_15m_2025['returns'].std(),

        df_15m_2025['volatility_30'].mean(),

        df_15m_2025['volatility_30'].max()
    ]
})

display(comparison_stats)

### Insight

The 15-minute timeframe exhibits higher volatility than the 1-minute timeframe.

This is expected because longer intervals aggregate more market movement into each observation.

Meanwhile, the 1-minute timeframe captures finer market microstructure and short-term fluctuations.

In [ ]:
high_vol = (
    df_1m_2025['volatility_30']
    >
    df_1m_2025['volatility_30'].quantile(0.90)
)

vol_threshold = (
    df_1m_2025['volatility_30']
    .quantile(0.90)
)

print(
    "High Volatility Periods:",
    high_vol.sum()
)

print(
    "Volatility Threshold:",
    vol_threshold
)

### Insight

High-volatility periods represent market regimes characterized by elevated uncertainty and stronger price fluctuations.

These periods are especially important for trading strategy design, as they may require stricter risk management and different trading behavior.

# Step 3 — Trading Strategy & Backtesting

After understanding market behavior through statistical and time-series analysis, a trading strategy was developed and evaluated.

### Objective

The purpose of this strategy is not maximizing profit, but rather demonstrating:

- Logical thinking
- Financial reasoning
- Strategy design
- Quantitative evaluation
- Backtesting methodology

### Strategy Type

A Long-Only Trend Following Strategy was selected using:

- EMA50 (Trend Filter)
- Volume Confirmation

The strategy enters the market only when:

1. Price is above EMA50 → bullish trend.
2. Trading volume exceeds its recent average → stronger conviction.

This reduces weak signals and aligns trades with trend direction.

In [ ]:
# ==================================================
# Strategy Indicators
# ==================================================

strategy_df = df_15m_2025.copy()

# Trend Indicator
strategy_df['EMA50'] = (
    strategy_df['close']
    .ewm(span=50, adjust=False)
    .mean()
)

# Average Volume
strategy_df['volume_ma20'] = (
    strategy_df['volume']
    .rolling(20)
    .mean()
)

strategy_df.dropna(inplace=True)

strategy_df.head()

### Why EMA50?

The Exponential Moving Average (EMA50) was selected as a trend indicator because it reacts faster to price changes than a simple moving average.

It helps identify:

- Trend direction
- Market momentum
- Potential entry timing

### Why Volume Confirmation?

High trading volume generally reflects stronger market participation.

Therefore, a trade is executed only when:

Current volume > Average volume

This helps reduce weak or low-conviction signals.

In [ ]:
# ==================================================
# Trading Signals
# ==================================================

strategy_df['signal'] = np.where(

    (
        strategy_df['close']
        >
        strategy_df['EMA50']
    )

    &

    (
        strategy_df['volume']
        >
        strategy_df['volume_ma20']
    ),

    1,
    0
)

strategy_df['position'] = (
    strategy_df['signal']
    .shift(1)
)

strategy_df[
    ['close', 'EMA50', 'signal']
].tail()

### Entry & Exit Logic

### Buy Signal

A position is opened when:

- Price > EMA50
- Volume > 20-period average volume

This suggests:

- Market trend is bullish
- Buying pressure is relatively strong

### Exit Signal

The position is closed when:

- Price falls below EMA50

This may indicate weakening momentum or trend reversal.

In [ ]:
# ==================================================
# Strategy Returns
# ==================================================

strategy_df['strategy_returns'] = (

    strategy_df['position']

    *

    strategy_df['returns']
)

strategy_df['equity_curve'] = (

    1
    +
    strategy_df['strategy_returns']

).cumprod()

strategy_df.head()

### Backtesting Methodology

The strategy was evaluated using historical Nasdaq Futures data for 2025.

The backtest simulates:

- Entry signals
- Exit signals
- Position holding
- Portfolio growth over time

Performance was measured using several financial metrics to evaluate risk and return behavior.

In [ ]:
# ==================================================
# Performance Metrics
# ==================================================

cumulative_return = (
    strategy_df['equity_curve'].iloc[-1]
    - 1
)

rolling_max = (
    strategy_df['equity_curve']
    .cummax()
)

drawdown = (

    strategy_df['equity_curve']
    -
    rolling_max

) / rolling_max

max_drawdown = drawdown.min()

trade_returns = strategy_df.loc[
    strategy_df['position'] == 1,
    'strategy_returns'
]

win_rate = (
    (trade_returns > 0)
    .mean()
) * 100

total_trades = (
    strategy_df['signal']
    .diff()
    .eq(1)
    .sum()
)

performance_summary = pd.DataFrame({

    'Metric': [

        'Cumulative Return',
        'Maximum Drawdown',
        'Win Rate',
        'Total Trades'

    ],

    'Value': [

        f'{cumulative_return:.2%}',
        f'{max_drawdown:.2%}',
        f'{win_rate:.2f}%',
        total_trades
    ]
})

display(performance_summary)

### Performance Interpretation

The strategy generated:

### Cumulative Return: 19.75%

This indicates positive portfolio growth during the testing period.

### Maximum Drawdown: -9.57%

The largest observed decline remained relatively controlled, suggesting moderate downside risk.

### Win Rate: 50.52%

Although the win rate is close to 50%, profitability can still be achieved because profitable trades may outweigh losing trades.

### Total Trades: 657

The strategy maintained sufficient market participation while avoiding excessive overtrading.

Overall, the strategy demonstrates reasonable performance while maintaining simplicity and interpretability.

In [ ]:
# ==================================================
# Risk Adjusted Return
# ==================================================

sharpe_ratio = (

    strategy_df['strategy_returns']
    .mean()

    /

    strategy_df['strategy_returns']
    .std()

) * np.sqrt(252 * 26)

print(
    "Sharpe Ratio:",
    round(sharpe_ratio, 2)
)

### Sharpe Ratio Interpretation

The Sharpe Ratio measures return relative to risk.

A value of 0.79 suggests moderate risk-adjusted performance.

This means the strategy generated returns, although there remains room for improving consistency and reducing volatility.

In [ ]:
fig = px.line(

    strategy_df,

    x='timestamps',
    y='equity_curve',

    title='Strategy Equity Curve',

    template='plotly_white'
)

fig.show()

### Equity Curve Insight

The equity curve demonstrates the evolution of portfolio value over time.

The general upward trend suggests that the strategy maintained positive long-term performance despite temporary drawdowns.

This indicates the strategy was capable of adapting reasonably well to changing market conditions.

In [ ]:
# ==================================================
# Buy / Sell Signals Visualization
# ==================================================

buy_signals = strategy_df[
    strategy_df['signal'].diff() == 1
]

sell_signals = strategy_df[
    strategy_df['signal'].diff() == -1
]

fig = go.Figure()

# Price
fig.add_trace(go.Scatter(
    x=strategy_df['timestamps'],
    y=strategy_df['close'],
    mode='lines',
    name='Close Price'
))

# Buy Signals
fig.add_trace(go.Scatter(
    x=buy_signals['timestamps'],
    y=buy_signals['close'],
    mode='markers',
    name='BUY',
    marker=dict(
        symbol='triangle-up',
        size=10
    )
))

# Sell Signals
fig.add_trace(go.Scatter(
    x=sell_signals['timestamps'],
    y=sell_signals['close'],
    mode='markers',
    name='SELL',
    marker=dict(
        symbol='triangle-down',
        size=10
    )
))

fig.update_layout(
    title='Trading Signals on Price Chart',
    template='plotly_white'
)

fig.show()

### Trading Signal Interpretation

The visualization highlights entry and exit points across market movements.

Buy signals generally appear during upward trends with stronger volume activity, while exit signals occur during weakening momentum.
This confirms that the strategy behaves consistently with its intended trend-following logic.

# Step 4 — Analytical Thinking

Beyond performance evaluation, it is important to understand the reasoning, strengths, and limitations behind the selected trading strategy.

The goal is not only profitability, but also:

- Interpretability
- Simplicity
- Robustness
- Risk awareness
- Future improvement potential

## 1. Why Was This Strategy Selected?

A simple trend-following strategy was intentionally selected to maintain interpretability and avoid unnecessary complexity.

The strategy combines:

### Trend Confirmation
Using EMA50 to identify market direction.

### Volume Confirmation
Using trading volume to filter weak signals and improve trade quality.

The rationale behind this design is that strong market moves are generally more reliable when supported by higher participation (volume) and aligned with the broader trend.

This approach also reduces overfitting risk and makes the strategy easier to explain and maintain.

## 2. What Is the Main Weakness?

The primary limitation of this strategy is its sensitivity to sideways markets.

During non-trending periods, price may frequently move above and below EMA50, generating false signals and unnecessary trades.

As a result:

- Trading frequency may increase
- Performance may weaken
- Small losses may accumulate

This is a common limitation of many trend-following systems.

## 3. How Could This Strategy Be Improved for Economic News and High Volatility?

Market volatility often increases significantly during:

- Federal Reserve announcements
- Inflation reports (CPI)
- Employment data (NFP)
- Major economic events

To improve robustness, the strategy could be enhanced by:

### Volatility Filter
Avoid trading during extremely high volatility periods.

### Economic Calendar Integration
Pause trading during major economic announcements to reduce unexpected market shocks.

### Dynamic Thresholds
Adjust entry conditions depending on market volatility regimes.

## 4. How Could Risk Management Be Improved?

Risk management could be strengthened using several mechanisms:

### Stop Loss
Automatically exit trades when losses exceed a predefined threshold.

### Take Profit
Secure gains before market reversals occur.

### Position Sizing
Adjust trade size depending on market volatility to reduce exposure during unstable periods.

### Maximum Daily Risk
Set a daily loss limit to avoid excessive drawdowns.

These improvements would likely increase strategy stability and reduce downside risk.

# Final Conclusion

This project analyzed Nasdaq Futures (NQ) market behavior using both 1-minute and 15-minute time-series data from 2025.

The analysis included:

- Data preprocessing
- Statistical analysis
- Time-series analysis
- Stationarity testing (ADF)
- Volatility analysis
- Feature engineering
- Trading strategy development
- Backtesting evaluation

A simple but interpretable trend-following strategy was implemented using EMA50 and volume confirmation.

Results showed:

- Positive cumulative return (19.75%)
- Controlled drawdown (-9.57%)
- Balanced win rate (50.52%)

While the strategy remains simple, it demonstrates a structured and explainable framework for quantitative market analysis and can serve as a foundation for future improvements.